# Winner Side Prediction Baseline

Train Random Forest and XGBoost models to predict whether the winner is represented as `R` or `B` using regular difference features from pre-fight career stats.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


RANDOM_STATE = 42
TEST_SIZE = 0.15
N_RANDOM_SPLITS = 5

data_path = Path("../data_clean/pre_fight_career_stats.csv")
df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Keep only fights with a red-side or blue-side winner.
df = df[df["winner_side"].isin(["R", "B"])].copy()
df = df.dropna(subset=["date", "winner_side"]).reset_index(drop=True)

print("Loaded:", data_path)
print("Shape:", df.shape)
print(df["winner_side"].value_counts())
df.head()

Loaded: ..\data_clean\pre_fight_career_stats.csv
Shape: (6396, 217)
winner_side
R    3957
B    2439
Name: count, dtype: int64


,fight_id,date,R_fighter,B_fighter,R_age,B_age,winner,winner_side,win_by,R_elo,...,B_DISTANCE_absorbed_per_min,B_DISTANCE_opp_att_per_min,B_CLINCH_land_per_min,B_CLINCH_att_per_min,B_CLINCH_absorbed_per_min,B_CLINCH_opp_att_per_min,B_GROUND_land_per_min,B_GROUND_att_per_min,B_GROUND_absorbed_per_min,B_GROUND_opp_att_per_min
0,8677,1994-12-16,Royce Gracie,Keith Hackney,28.010951,36.670773,Royce Gracie,R,Submission,1589.228905,...,0.000000,1.008403,1.008403,1.512605,0.000000,0.000000,13.109244,19.663866,0.504202,1.512605
1,8678,1994-12-16,Royce Gracie,Dan Severn,28.010951,36.522930,Royce Gracie,R,Submission,1605.295686,...,1.142857,1.714286,0.000000,0.000000,1.714286,1.714286,0.000000,0.000000,1.714286,1.714286
2,8660,1995-07-14,Ken Shamrock,Dan Severn,31.419576,37.097878,Ken Shamrock,R,Submission,1544.184241,...,0.145808,0.400972,0.036452,0.072904,0.182260,0.291616,0.583232,1.057108,0.109356,0.109356
3,8661,1995-07-14,Oleg Taktarov,Dave Beneteau,27.882272,NaN,Oleg Taktarov,R,Submission,1522.336486,...,0.000000,0.000000,0.359281,0.898204,0.179641,0.359281,3.592814,5.568862,0.179641,0.179641
4,8663,1995-07-14,Oleg Taktarov,Anthony Macias,27.882272,26.422998,Oleg Taktarov,R,Submission,1542.390681,...,0.000000,0.000000,1.714286,1.714286,0.000000,0.000000,1.714286,1.714286,0.000000,0.000000


## Balance Side Labels

Each fight can be represented in either orientation. If a fight is swapped, every matching `R_...` and `B_...` column changes sides, `R_fighter` and `B_fighter` change sides, and `winner_side` flips. This balances the label without duplicating fights.

In [2]:
def find_red_blue_pairs(columns):
    pairs = []
    for col in columns:
        if col.startswith("R_"):
            b_col = "B_" + col[2:]
            if b_col in columns:
                pairs.append((col, b_col))
    return pairs


def swap_red_blue_rows(data, swap_mask):
    balanced = data.copy()
    pairs = find_red_blue_pairs(balanced.columns)
    r_cols = [r for r, _ in pairs]
    b_cols = [b for _, b in pairs]

    r_values = balanced.loc[swap_mask, r_cols].to_numpy(copy=True)
    b_values = balanced.loc[swap_mask, b_cols].to_numpy(copy=True)
    balanced.loc[swap_mask, r_cols] = b_values
    balanced.loc[swap_mask, b_cols] = r_values

    balanced.loc[swap_mask, "winner_side"] = balanced.loc[swap_mask, "winner_side"].map({"R": "B", "B": "R"})

    return balanced


def randomly_balance_winner_side(data, random_state=42, balance_by_year=True):
    rng = np.random.default_rng(random_state)
    balanced = data.copy().reset_index(drop=True)

    desired_side = pd.Series(index=balanced.index, dtype="object")
    groups = balanced["date"].dt.year if balance_by_year else pd.Series(0, index=balanced.index)

    for _, group_idx in balanced.groupby(groups).groups.items():
        group_idx = np.array(list(group_idx))
        n_rows = len(group_idx)
        group_sides = ["R"] * (n_rows // 2) + ["B"] * (n_rows // 2)

        if n_rows % 2 == 1:
            group_sides.append(rng.choice(["R", "B"]))

        group_sides = np.array(group_sides)
        rng.shuffle(group_sides)
        desired_side.loc[group_idx] = group_sides

    swap_mask = balanced["winner_side"].to_numpy() != desired_side.to_numpy()
    balanced = swap_red_blue_rows(balanced, swap_mask)
    balanced["target"] = (balanced["winner_side"] == "R").astype(int)

    return balanced


balanced_df = randomly_balance_winner_side(df, random_state=RANDOM_STATE)

print("Original target distribution:")
print(df["winner_side"].value_counts())
print("\nBalanced target distribution:")
print(balanced_df["winner_side"].value_counts())

balanced_df.sort_values("date").groupby(balanced_df["date"].dt.year)["winner_side"].value_counts(normalize=True).unstack().tail(10)

Original target distribution:
winner_side
R    3957
B    2439
Name: count, dtype: int64

Balanced target distribution:
winner_side
B    3199
R    3197
Name: count, dtype: int64


winner_side,B,R
date,,
2017,0.500000,0.500000
2018,0.500000,0.500000
2019,0.500000,0.500000
2020,0.501458,0.498542
2021,0.500000,0.500000
2022,0.498807,0.501193
2023,0.500000,0.500000
2024,0.501166,0.498834
2025,0.500000,0.500000


## Build Model Features

Outcome and identity columns are excluded from features. Matching `R_...` and `B_...` stats are also turned into difference features.

In [3]:
def add_matchup_features(data):
    diff_features = {}

    for r_col, b_col in find_red_blue_pairs(data.columns):
        suffix = r_col[2:]
        if pd.api.types.is_numeric_dtype(data[r_col]) and pd.api.types.is_numeric_dtype(data[b_col]):
            if suffix == "age":
                center = pd.concat([data[r_col], data[b_col]], ignore_index=True).median()
                diff_features["diff_age_centered_sq"] = (
                    (data[r_col] - center) ** 2
                    - (data[b_col] - center) ** 2
                )
            else:
                diff_features[f"DIFF_{suffix}"] = data[r_col] - data[b_col]

    diff_df = pd.DataFrame(diff_features, index=data.index)
    return pd.concat([data.copy(), diff_df], axis=1)


model_df = add_matchup_features(balanced_df)

leakage_cols = [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "winner",
    "winner_side",
    "win_by",
]

feature_cols = [
    col for col in model_df.columns
    if (
        col.startswith("DIFF_")
        or col == "diff_age_centered_sq"
    )
    and pd.api.types.is_numeric_dtype(model_df[col])
]


def unique_preserve_order(cols):
    return list(dict.fromkeys(cols))


def diff_suffix(feature_name):
    if feature_name == "diff_age_centered_sq":
        return "age"
    if feature_name.startswith("DIFF_"):
        return feature_name.replace("DIFF_", "", 1)
    return feature_name


shared_suffixes = {
    "age",
    "elo",
    "reach",
    "height",
    "total_fights",
    "total_fight_time",
}

shared_feature_cols = [
    col for col in feature_cols
    if diff_suffix(col) in shared_suffixes
]

per_min_feature_cols = [
    col for col in feature_cols
    if col.startswith("DIFF_") and diff_suffix(col).endswith("_per_min")
]

cumulative_feature_cols = [
    col for col in feature_cols
    if col.startswith("DIFF_") and diff_suffix(col).endswith("_total")
]

feature_sets = {
    "All regular diff features": feature_cols,
    "Per-minute stats + shared context": unique_preserve_order(shared_feature_cols + per_min_feature_cols),
    "Cumulative totals + shared context": unique_preserve_order(shared_feature_cols + cumulative_feature_cols),
}

y = model_df["target"]

print("Rows:", len(model_df))
print("Features:", len(feature_cols))
print("Shared context features:", shared_feature_cols)
pd.DataFrame(
    {"feature_set": name, "n_features": len(cols)}
    for name, cols in feature_sets.items()
)

Rows: 6396
Features: 105
Shared context features: ['diff_age_centered_sq', 'DIFF_elo', 'DIFF_total_fights', 'DIFF_total_fight_time']


,feature_set,n_features
0,All regular diff features,105
1,Per-minute stats + shared context,43
2,Cumulative totals + shared context,52


## Random Split Model Runs

Evaluate regular-difference features with repeated random stratified train/test splits.

In [4]:
def make_rf_pipeline(random_state):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=600,
                    min_samples_leaf=8,
                    max_features="sqrt",
                    n_jobs=-1,
                    random_state=random_state,
                    class_weight=None,
                ),
            ),
        ]
    )


def make_xgb_pipeline(random_state):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBClassifier(
                    n_estimators=500,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=random_state,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def evaluate_model(feature_set_name, cols, model_name, make_pipeline):
    X_subset = model_df[cols].replace([np.inf, -np.inf], np.nan)
    splitter = StratifiedShuffleSplit(
        n_splits=N_RANDOM_SPLITS,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    split_rows = []
    for split_num, (train_idx, test_idx) in enumerate(splitter.split(X_subset, y), start=1):
        model = make_pipeline(RANDOM_STATE + split_num)

        model.fit(X_subset.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X_subset.iloc[test_idx])
        proba = model.predict_proba(X_subset.iloc[test_idx])[:, 1]

        split_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "split": split_num,
            "n_features": len(cols),
            "accuracy": accuracy_score(y.iloc[test_idx], pred),
            "roc_auc": roc_auc_score(y.iloc[test_idx], proba),
        })

    return split_rows


y = model_df["target"]

model_runs = {
    "Random Forest": make_rf_pipeline,
    "XGBoost": make_xgb_pipeline,
}

split_results = pd.DataFrame(
    row
    for feature_set_name, cols in feature_sets.items()
    for model_name, make_pipeline in model_runs.items()
    for row in evaluate_model(feature_set_name, cols, model_name, make_pipeline)
)

model_summary = (
    split_results
    .groupby(["feature_set", "model"])
    .agg(
        n_features=("n_features", "first"),
        n_splits=("split", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
    )
    .sort_values("mean_accuracy", ascending=False)
    .reset_index()
)

selected_feature_set_name = model_summary.loc[0, "feature_set"]
selected_feature_cols = feature_sets[selected_feature_set_name]
X_selected = model_df[selected_feature_cols].replace([np.inf, -np.inf], np.nan)

print("Best feature set:", selected_feature_set_name)
print("Selected feature count:", len(selected_feature_cols))
print("Uses diff_age_centered_sq:", "diff_age_centered_sq" in selected_feature_cols)
model_summary

Best feature set: All regular diff features
Selected feature count: 105
Uses diff_age_centered_sq: True


,feature_set,model,n_features,n_splits,mean_accuracy,std_accuracy,mean_roc_auc,std_roc_auc
0,All regular diff features,Random Forest,105,5,0.602292,0.016094,0.633536,0.014645
1,Per-minute stats + shared context,Random Forest,43,5,0.602083,0.018458,0.628641,0.018083
2,Per-minute stats + shared context,XGBoost,43,5,0.598542,0.003939,0.628323,0.012950
3,Cumulative totals + shared context,Random Forest,52,5,0.594375,0.011845,0.620682,0.009488
4,All regular diff features,XGBoost,105,5,0.594375,0.011520,0.630156,0.010342
5,Cumulative totals + shared context,XGBoost,52,5,0.585625,0.019410,0.614154,0.011976


## Per-Split Results

Each split is random and stratified, so every test set keeps the same balanced target mix.

In [5]:
split_results.sort_values(["feature_set", "model", "split"]).reset_index(drop=True)

,feature_set,model,split,n_features,accuracy,roc_auc
0,All regular diff features,Random Forest,1,105,0.618750,0.649327
1,All regular diff features,Random Forest,2,105,0.605208,0.640829
2,All regular diff features,Random Forest,3,105,0.615625,0.641432
3,All regular diff features,Random Forest,4,105,0.590625,0.620595
4,All regular diff features,Random Forest,5,105,0.581250,0.615495
5,All regular diff features,XGBoost,1,105,0.614583,0.641914
6,All regular diff features,XGBoost,2,105,0.589583,0.624826
7,All regular diff features,XGBoost,3,105,0.586458,0.639818
8,All regular diff features,XGBoost,4,105,0.588542,0.626493
9,All regular diff features,XGBoost,5,105,0.592708,0.617730


In [6]:
final_importances = []

for model_name, make_pipeline in model_runs.items():
    model = make_pipeline(RANDOM_STATE)
    model.fit(X_selected, y)
    fitted_model = model.named_steps["model"]

    if hasattr(fitted_model, "feature_importances_"):
        model_importances = pd.DataFrame({
            "model": model_name,
            "feature_set": selected_feature_set_name,
            "feature": selected_feature_cols,
            "importance": fitted_model.feature_importances_,
        })
        final_importances.append(model_importances)

importances = (
    pd.concat(final_importances, ignore_index=True)
    .sort_values(["model", "importance"], ascending=[True, False])
    .reset_index(drop=True)
)

importances.groupby("model").head(25)

,model,feature_set,feature,importance
0,Random Forest,All regular diff features,DIFF_elo,0.027996
1,Random Forest,All regular diff features,DIFF_TOTAL_STR_absorbed_per_min,0.020018
2,Random Forest,All regular diff features,DIFF_HEAD_absorbed_per_min,0.019594
3,Random Forest,All regular diff features,DIFF_SIG_STR_def,0.016850
4,Random Forest,All regular diff features,DIFF_TD_land_per_min,0.015925
5,Random Forest,All regular diff features,DIFF_TOTAL_STR_opp_att_per_min,0.015100
6,Random Forest,All regular diff features,DIFF_GROUND_absorbed_per_min,0.014724
7,Random Forest,All regular diff features,DIFF_TD_att_per_min,0.014665
8,Random Forest,All regular diff features,DIFF_SIG_STR_acc,0.014449
9,Random Forest,All regular diff features,DIFF_SIG_STR_absorbed_per_min,0.014160
